# Create AAV6-ML plots for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 01/04/2026

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2

## 2. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'Tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for plots
plots_dir = general_dir/'Figures'
os.makedirs(plots_dir, exist_ok=True)

In [ ]:
tables_dir

## 3. Define Functions
### 3.1. Helper functions

In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)
    
    

### 3.2. Script Functions

In [ ]:
# Function to create distribution ecdf figure of one tissue/ext type with hue between mouse_ID

def distribution_ecdf(table, tissue, extraction, column, save=False, output_path=None):
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ].copy()
    if column == 'Proportion':
        df = df[df['Pseudo'] == 0].copy()

    df = df[df[column] > 0].copy()


    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    sns.ecdfplot(
        data=df,
        x=column,
        hue="Mouse_ID",
        ax=ax,
        linewidth=2
    )
    if column == 'Log2_enrichment':
        ax.axvline(1, linestyle="--", linewidth=1.5, color="red", alpha=0.9)
        ax.axvline(df[column].mean(), linestyle="--", linewidth=1.5, color="blue", alpha=0.9)
        
    ax.set_xlim(df[column].min(), df[column].max())
    ax.set_xscale("log")
    
    ax.set_xlabel(f"Variant {column} in sample")
    ax.set_ylabel("Cumulative fraction of AA sequences")
    ax.set_title(f"{extraction} {tissue}: cumulative distribution of AA-sequence {column}")

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()



In [ ]:
distribution_ecdf (df_long_table, 'heart', 'cDNA', 'Proportion')

In [ ]:
distribution_ecdf (df_long_table, 'liver', 'gDNA', 'Log2_enrichment')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def proportion_distribution(table, tissue, extraction, prop_col="Proportion",
                            bins=60, min_prop=1e-7, max_prop=None,
                            save=False, output_path=None):
    """
    Plot overlaid proportion distributions for all Mouse_IDs within one tissue and extraction type.

    Parameters
    ----------
    table : pd.DataFrame
        Long-format table containing at least:
        ['Tissue', 'Extraction_type', 'Mouse_ID', prop_col]
    tissue : str
        Tissue to plot, e.g. 'heart' or 'liver'
    extraction : str
        Extraction type to plot, e.g. 'cDNA' or 'gDNA'
    prop_col : str
        Column containing the proportions
    bins : int
        Number of histogram bins
    min_prop : float
        Lower x-limit and lower bound for filtering
    max_prop : float or None
        Upper x-limit; if None, inferred from data
    save : bool
        Whether to save the figure
    output_path : str or Path or None
        Save path if save=True
    """

    # -----------------------------
    # Filter data
    # -----------------------------
    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ].copy()

    if df.empty:
        raise ValueError(f"No rows found for Tissue='{tissue}' and Extraction_type='{extraction}'")

    # remove zeros / negative values for log-scale plotting
    df = df[df[prop_col] > 0].copy()

    if df.empty:
        raise ValueError(f"No positive values found in column '{prop_col}' for {tissue} / {extraction}")

    if max_prop is None:
        max_prop = df[prop_col].max()

    # log-spaced bins
    bin_edges = np.logspace(np.log10(min_prop), np.log10(max_prop), bins)

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # preserve mouse order if possible
    mouse_order = ["f1", "f2", "f3", "m1", "m2", "m3"]
    present_mice = [m for m in mouse_order if m in df["Mouse_ID"].unique()]

    # plot one histogram per mouse
    for mouse_id in present_mice:
        sub = df[df["Mouse_ID"] == mouse_id]

        ax.hist(
            sub[prop_col],
            bins=bin_edges,
            alpha=0.35,
            label=mouse_id,
            edgecolor="none"
        )

    # threshold line
    ax.axvline(1e-6, linestyle="--", linewidth=1.5, color="red", alpha=0.05)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(min_prop, max_prop)

    ax.set_xlabel("Variant proportion in sample")
    ax.set_ylabel("Number of AA sequences")
    ax.set_title(f"{extraction} {tissue}: distribution of AA-sequence proportions")

    # cleaner legend
    ax.legend(title="Mouse ID", frameon=False, ncol=2)

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
proportion_distribution(df_long_table, 'heart', 'cDNA')

## 4. Script
### 4.1. Load CSV Files

In [ ]:
%%time
#load long table
df_long_table = read_csv_file(tables_dir / 'summary', "df_long_table_metadata")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_pooled_table")

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long_table, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

### 4.3. Load tissue/ext specific librarys

In [ ]:
dict_library = {}
for tissue, ext in product(TISSUE, EXT):
    df = read_csv_file(tables_dir/tissue, f'df_input_library_for_{ext}_{tissue}')
    dict_library.setdefault(tissue,{})[ext] = df

# Load raw library
# Raw library laden um sie mit den spezifischen zu correlieren